# Learning Clustering: Understanding Dendrograms

This notebook introduces **hierarchical clustering** and shows how to read a **dendrogram**.

By the end, you should understand:

- What clustering means
- What hierarchical clustering does
- How a dendrogram is built
- How to choose the number of clusters from a dendrogram

## 1. What Is Clustering?

Clustering is an **unsupervised machine learning** technique.

That means we do not give the model labels like `Class A` or `Class B`. Instead, the algorithm tries to find groups based on similarity.

Example:

- Customers with similar buying habits
- Students with similar marks
- Cities with similar weather patterns
- Documents with similar words

## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

plt.style.use("seaborn-v0_8-whitegrid")

## 3. Create a Simple Dataset

We will create artificial data with visible groups. This makes it easier to understand what the clustering algorithm is doing.

In [ ]:
X, true_labels = make_blobs(
    n_samples=40,
    centers=3,
    cluster_std=1.0,
    random_state=42
)

df = pd.DataFrame(X, columns=["Feature_1", "Feature_2"])
df.head()

## 4. Visualize the Data

Before clustering, always look at the data if it has only two or three features.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["Feature_1"], df["Feature_2"], s=70)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Original Data Points")
plt.show()

## 5. Standardize the Data

Many clustering methods depend on distance. If one feature has much larger values than another, it can dominate the distance calculation.

Standardization puts features on a similar scale.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

pd.DataFrame(X_scaled, columns=df.columns).head()

## 6. Hierarchical Clustering

Hierarchical clustering builds clusters step by step.

In **agglomerative hierarchical clustering**:

1. Every data point starts as its own cluster.
2. The two closest clusters are merged.
3. This keeps repeating until all points are in one big cluster.

The history of these merges is shown using a **dendrogram**.

## 7. Create the Linkage Matrix

The linkage matrix stores which clusters were merged and at what distance.

Common linkage methods:

- `single`: distance between nearest points
- `complete`: distance between farthest points
- `average`: average distance between points
- `ward`: minimizes within-cluster variance

In [ ]:
linked = linkage(X_scaled, method="ward")

linked[:5]

## 8. Plot the Dendrogram

How to read it:

- Each leaf at the bottom is one data point.
- Branches show clusters being joined.
- The height of a merge shows the distance between clusters.
- A big vertical jump means two very different clusters were merged.

In [ ]:
plt.figure(figsize=(12, 6))
dendrogram(linked)
plt.title("Dendrogram Using Ward Linkage")
plt.xlabel("Data Point Index")
plt.ylabel("Distance")
plt.show()

## 9. Choosing the Number of Clusters

A common method is to look for the **largest vertical gap** in the dendrogram.

Imagine drawing a horizontal line across the dendrogram. The number of vertical branches crossed by the line is the number of clusters.

In the next cell, change `cut_distance` and observe how the result changes.

In [ ]:
cut_distance = 5

plt.figure(figsize=(12, 6))
dendrogram(linked)
plt.axhline(y=cut_distance, color="red", linestyle="--", label=f"Cut distance = {cut_distance}")
plt.title("Dendrogram with Cut Line")
plt.xlabel("Data Point Index")
plt.ylabel("Distance")
plt.legend()
plt.show()

## 10. Assign Cluster Labels

Now we convert the dendrogram cut into actual cluster labels.

In [ ]:
cluster_labels = fcluster(linked, t=cut_distance, criterion="distance")

df_clustered = df.copy()
df_clustered["Cluster"] = cluster_labels

df_clustered.head()

## 11. Visualize the Final Clusters

In [ ]:
plt.figure(figsize=(7, 5))

for cluster_id in sorted(df_clustered["Cluster"].unique()):
    cluster_data = df_clustered[df_clustered["Cluster"] == cluster_id]
    plt.scatter(
        cluster_data["Feature_1"],
        cluster_data["Feature_2"],
        s=80,
        label=f"Cluster {cluster_id}"
    )

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Clusters from Hierarchical Clustering")
plt.legend()
plt.show()

## 12. Try Different Linkage Methods

Run this cell and compare the dendrogram shapes.

In [ ]:
methods = ["single", "complete", "average", "ward"]

plt.figure(figsize=(14, 10))

for i, method in enumerate(methods, start=1):
    linked_method = linkage(X_scaled, method=method)
    plt.subplot(2, 2, i)
    dendrogram(linked_method, no_labels=True)
    plt.title(f"{method.title()} Linkage")
    plt.xlabel("Data Points")
    plt.ylabel("Distance")

plt.tight_layout()
plt.show()

## 13. Mini Exercises

Try these changes to strengthen your understanding:

1. Change `cut_distance` to `3`, `4`, `6`, and `8`. How many clusters do you get?
2. Change `n_samples` from `40` to `80`. Does the dendrogram become harder to read?
3. Change `centers=3` to `centers=4`. Can the dendrogram reveal four groups?
4. Compare `single`, `complete`, `average`, and `ward`. Which one gives the clearest dendrogram?

## 14. Key Takeaways

- A dendrogram shows the full merging history of hierarchical clustering.
- The y-axis represents distance or dissimilarity.
- Large vertical jumps suggest meaningful separation between clusters.
- A horizontal cut line helps decide the number of clusters.
- Different linkage methods can produce different cluster structures.